In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    sum,round
    
)

In [2]:
spark=(
    SparkSession.builder
    .appName("Sales trend Analysis")
    .master("local[*]")
    .getOrCreate()
    
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/13 17:59:56 WARN Utils: Your hostname, MacBook-pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.174 instead (on interface en0)
26/06/13 17:59:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 17:59:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/13 17:59:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
sales_df=spark.read.parquet(
    "output/gold/sales_fact"
)
sales_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- purchase_timestamp: timestamp (nullable = true)
 |-- purchase_year: integer (nullable = true)
 |-- purchase_month: integer (nullable = true)
 |-- purchase_day: integer (nullable = true)
 |-- purchase_quarter: integer (nullable = true)
 |-- purchase_weekday: string (nullable = true)



calculating How are sales changing month-over-month 

In [7]:
# calculating monthly sales dataframe

monthly_sales_df =(
    sales_df
    .groupBy (
        "purchase_year",
        "purchase_month"
    )
    .agg(
        round(sum("payment_value"),2).alias("monthly_revenue"))
).orderBy(
    "purchase_year",
    "purchase_month"
)
print("\nMonthly Revenue")

monthly_sales_df.show(50,truncate=False)


Monthly Revenue
+-------------+--------------+---------------+
|purchase_year|purchase_month|monthly_revenue|
+-------------+--------------+---------------+
|2016         |9             |347.52         |
|2016         |10            |73914.58       |
|2016         |12            |19.62          |
|2017         |1             |187779.41      |
|2017         |2             |344134.79      |
|2017         |3             |526961.66      |
|2017         |4             |505665.53      |
|2017         |5             |724504.55      |
|2017         |6             |600753.27      |
|2017         |7             |737293.08      |
|2017         |8             |870105.9       |
|2017         |9             |1015849.57     |
|2017         |10            |1021169.27     |
|2017         |11            |1583869.01     |
|2017         |12            |1042855.86     |
|2018         |1             |1408365.65     |
|2018         |2             |1306048.8      |
|2018         |3             |1475599.95   

In [ ]:
# saving then Monthly sales
monthly_sales_df.write.mode(
    "overwirte"
).parquet("output/analyatics/monthly_sales")

print("\nSales Trend Analysis Completed")